In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import scienceplots
plt.style.use(['science', 'notebook'])
plt.rcParams.update({
    "text.usetex": False,
})

from mg_sampling import PROJECT_DIR
from mg_sampling.tools import to_wsv, edges_to_range
from mg_sampling.classes import MetadSim
from mg_sampling.plot import LABELS
from mg_sampling.bias import fit_bias, associative_loss_fn, bias_fn, ASSOCIATIVE_P_0, ASSOCIATIVE_BOUNDS, PARAMS_NAMES, dissociative_loss_fn, DISSOCIATIVE_BOUNDS, DISSOCIATIVE_P_0
from mg_sampling.constants import π

### ECC/SPCE: dissociative pathway fit: doesn't work

In [ ]:
otypes = ['op']
forcefields = ['ecc_spce']

sims = {
    (ot, ff): MetadSim(f'{PROJECT_DIR}/metad/{ot}_{ff}', f'{ot}_{ff}')
              for ff in forcefields
              for ot in otypes
}

for sim in sims.values():
    params = fit_bias(sim, bias_fn, associative_loss_fn, ASSOCIATIVE_P_0, bounds=ASSOCIATIVE_BOUNDS)
    to_wsv(pd.DataFrame([params], columns=PARAMS_NAMES), f'{sim.path}/bias_params.wsv')
    print(params)

## Plot fitted biases

In [ ]:
from mg_sampling.tools import read_wsv
from mg_sampling.plot import plot_bias, plot_bias_XY_marginals
from mg_sampling.bias import dissociative_mask, associative_mask


otypes = ["op"]
#forcefields = ["amber_tip3p", "mMg_tip3p", "nMg_tip3p", "mMg_opc"]
forcefields = ["ecc_spce"]

x_edges, y_edges = np.linspace(0, 2, 61), np.linspace(4, 7, 91)
X_edges, Y_edges = np.linspace(0, 2, 61), np.linspace(-1, 2, 91)

sims = {
    (ot, ff): MetadSim(f'{PROJECT_DIR}/metad/{ot}_{ff}', f'{ot}_{ff}')
              for ff in forcefields
              for ot in otypes
}
for (ot, ff), sim in sims.items():
    params = read_wsv(f'{sim.path}/bias_params.wsv').iloc[0]
    #params = (12, 0.1, -0.3)
    fig = plt.figure(layout='constrained', figsize=(10, 6))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    ax = subfigs[0].subplots()
    plot_bias(sim, bias_fn, params, bins=(x_edges, y_edges), ax=ax)
    ax1, ax2 = subfigs[1].subplots(nrows=2)
    plot_bias_XY_marginals(sim, bias_fn, params, mask=associative_mask, X_bins=X_edges, Y_bins=Y_edges, axes=(ax1, ax2))

    fig.suptitle(f"{LABELS[ff]}: {LABELS[ot]}", fontsize=16)
    ax1.legend()
    fig.savefig(f"figures/biasfit_{ot}_{ff}.pdf")

### OP-fitted bias on OB

In [ ]:
otypes = ['ob']
forcefields = ["amber_tip3p", "mMg_tip3p", "nMg_tip3p", "mMg_opc"]

sims = {
    (ot, ff): MetadSim(f'{PROJECT_DIR}/metad/{ot}_{ff}', f'{ot}_{ff}')
              for ff in forcefields
              for ot in otypes
}
for (ot, ff), sim in sims.items():
    params = read_wsv(f'{PROJECT_DIR}/metad/op_{ff}/bias_params.wsv').iloc[0]
    fig = plt.figure(layout='constrained', figsize=(10, 6))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    ax = subfigs[0].subplots()
    plot_bias(sim, params, ax)
    ax1, ax2 = subfigs[1].subplots(nrows=2)
    plot_bias_marginals(sim, params, (ax1, ax2))

    fig.suptitle(f"{LABELS[ff]}: {LABELS[ot]} ({LABELS['op']} fit)", fontsize=16)
    ax1.legend()
    fig.savefig(f"figures/biasfit_opfit_{ot}_{ff}.pdf")